Step 1:
Create a new Notebook name: Delta_Assignment. 

Step 2: Create Dataset named: customer_master.csv and customer_incremental.csv

Step 3: Upload Both Files
Catalog → workspace → default → Volumes → superstore
Upload:

customer_master.csv
customer_incremental.csv

### **Read Master Dataset**

In [0]:
master_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/superstore/customer_master.csv")

display(master_df)

CustomerID,Name,City,Age,Email
101,Aman,Delhi,24,aman@gmail.com
102,Priya,Mumbai,27,priya@gmail.com
103,Rahul,Chandigarh,30,rahul@gmail.com
104,Simran,Amritsar,26,simran@gmail.com
105,Karan,Jaipur,29,karan@gmail.com


### **Handle Missing Values**

In [0]:
master_df = master_df.fillna("Unknown")

### **Remove Duplicate**

In [0]:
master_df = master_df.dropDuplicates()

In [0]:
print("Rows:", master_df.count())


Rows: 5


### Save as Delta Table

In [0]:
master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/superstore/customer_delta")

### Read Delta Table

In [0]:
from delta.tables import DeltaTable

delta_df = spark.read.format("delta") \
.load("/Volumes/workspace/default/superstore/customer_delta")

display(delta_df)

CustomerID,Name,City,Age,Email
101,Aman,Delhi,24,aman@gmail.com
103,Rahul,Chandigarh,30,rahul@gmail.com
104,Simran,Amritsar,26,simran@gmail.com
105,Karan,Jaipur,29,karan@gmail.com
102,Priya,Mumbai,27,priya@gmail.com


### Read Incremental Dataset

In [0]:
incremental_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/superstore/customer_incremental.csv")

display(incremental_df)

CustomerID,Name,City,Age,Email
102,Priya,Pune,28,priya@gmail.com
105,Karan,Jaipur,30,karan@gmail.com
106,Neha,Delhi,25,neha@gmail.com
107,Rohit,Lucknow,31,rohit@gmail.com


### Perform MERGE

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/workspace/default/superstore/customer_delta"
)

deltaTable.alias("old").merge(
    incremental_df.alias("new"),
    "old.CustomerID = new.CustomerID"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Validate

In [0]:
final_df = spark.read.format("delta") \
.load("/Volumes/workspace/default/superstore/customer_delta")

display(final_df)

CustomerID,Name,City,Age,Email
101,Aman,Delhi,24,aman@gmail.com
103,Rahul,Chandigarh,30,rahul@gmail.com
104,Simran,Amritsar,26,simran@gmail.com
102,Priya,Pune,28,priya@gmail.com
105,Karan,Jaipur,30,karan@gmail.com
106,Neha,Delhi,25,neha@gmail.com
107,Rohit,Lucknow,31,rohit@gmail.com


### Count Rows

In [0]:
print("Total Rows:", final_df.count())

Total Rows: 7


### Check Duplicates

In [0]:
print("Duplicate Rows:",
      final_df.count() - final_df.dropDuplicates().count())

Duplicate Rows: 0


### Save Final Delta Table as CSV

In [0]:
final_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/superstore/final_customer_data")